In [ ]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

import math
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from collections import Counter

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Preprocessing
from sklearn.preprocessing import MinMaxScaler, StandardScaler, MaxAbsScaler, RobustScaler, PowerTransformer, QuantileTransformer, OrdinalEncoder, LabelEncoder
from sklearn.impute import SimpleImputer

# Model Selection
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split, GridSearchCV, RepeatedStratifiedKFold
#import autogluon as ag

# Models
from sklearn.ensemble import HistGradientBoostingClassifier, GradientBoostingClassifier, AdaBoostClassifier,RandomForestClassifier,ExtraTreesClassifier,VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import optuna
from optuna.integration import LightGBMPruningCallback

import torch
import torch.nn as nn
import torch.optim as optim
if torch.cuda.is_available():
    device = torch.device("cuda")  
else:
    device = torch.device("cpu")   
    
# Metrics 
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.metrics import auc
from sklearn.metrics import roc_auc_score, roc_curve, make_scorer, f1_score

import warnings
warnings.filterwarnings('ignore')

In [ ]:
train = pd.read_csv('/kaggle/input/playground-series-s3e23/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s3e23/test.csv')
sample_submission = pd.read_csv('/kaggle/input/playground-series-s3e23/sample_submission.csv',index_col='id')

In [ ]:
train.drop('id',axis=1,inplace=True)
test.drop('id',axis=1,inplace=True)

print('The shape of the train data:', train.shape)
print('The shape of the test data:', test.shape)

In [ ]:
num_var = [column for column in train.columns if train[column].nunique() > 10]
target = 'defects'

In [ ]:
learningMetadata = {}

In [ ]:
def auc_score(y_true, y_pred):
    return roc_auc_score(y_true, y_pred)

In [ ]:
def learner(df_train,df_test, model, selected_features, note):
    skf = StratifiedKFold(shuffle=True, random_state=1)
    auc_accu_stratified = []
    model_preds = []
    
    X_train = df_train[selected_features]
    Y_train = df_train[target]
    X_test = df_test[selected_features]
    
    for i,(train_index, test_index) in enumerate(skf.split(X_train, Y_train)):
        print('---------------------------------------------------------------')
        x_train_fold, x_test_fold = X_train.iloc[train_index], X_train.iloc[test_index]
        y_train_fold, y_test_fold = Y_train[train_index], Y_train[test_index]
       
        model.fit(x_train_fold, y_train_fold)
        aucScore = auc_score(y_test_fold, model.predict_proba(x_test_fold)[:,1])
        testPreds = model.predict_proba(X_test)
        model_preds.append(testPreds)
        print(f"Fold, {i+1}, ==> {note} oof F1 score is ==> {aucScore}")
        auc_accu_stratified.append(aucScore)
    
    print('---------------------------------------------------------------')
    avgAcc = np.array(auc_accu_stratified).mean()
    print('Average Accuracy of model is:', avgAcc)
    learningMetadata[note] = {
        'cols': selected_features,
        'avgAcc': avgAcc
    }
    
    return model,model_preds,avgAcc

In [ ]:
if False:
    rf,preds = learner(train,test,RandomForestClassifier(),num_var,'RandomForestBasicAllCols')

In [ ]:
def subm(preds, file):
    sample_submission['defects'] = preds
    sample_submission.to_csv('sub'+file+'.csv')

In [ ]:
def rf_feat_importance(m, df):
    return pd.DataFrame({'cols':df.columns, 'imp':m.feature_importances_}
                       ).sort_values('imp', ascending=False)

In [ ]:
if False:
    fi = rf_feat_importance(rf, train[num_var])
    fi[:10]

In [ ]:
def plot_fi(fi):
    return fi.plot('cols', 'imp', 'barh', figsize=(12,7), legend=False)


if False:
    plot_fi(fi[:30]);

In [ ]:
X=train[num_var]
y=train[target]

In [ ]:
nfolds = 5
skfold = StratifiedKFold(n_splits=nfolds,shuffle=True,random_state=0)

In [ ]:
def rf_objective(trial):

        # Set the hyperparameters of the XGBoost classifier.
        params = { 
            'n_estimators' : trial.suggest_int("n_estimators", 20, 500),
            'max_depth' : trial.suggest_int("max_depth", 2, 15),
            'min_samples_split' : trial.suggest_int("min_samples_split", 20, 1000),
            'min_samples_leaf' : trial.suggest_int("min_samples_leaf", 20, 1000),
            'max_features':trial.suggest_float('max_features',0.01,1),
            }
        
        rf_auc_score_avg = 0
        for idx, (train_idx,val_idx) in enumerate(skfold.split(X,y)):
            train_X = X.iloc[train_idx]
            val_X = X.iloc[val_idx]
            train_y = y[train_idx]
            val_y = y[val_idx]

            rf_model = RandomForestClassifier(**params)

            rf_model.fit(train_X,train_y)

            rf_prediction = rf_model.predict_proba(val_X)[:,1]
            rf_auc_score = auc_score(val_y, rf_prediction)
#             print(f'The AUC score evaluated on the validation subset using XGB model for fold {idx}: ', xgb_auc_score)

            rf_auc_score_avg += rf_auc_score
            
        rf_auc_score_avg /=nfolds
#         print(f'The averaged AUC score evaluated on the validation subset using XGB model:', rf_auc_score_avg)
        return -rf_auc_score_avg

In [ ]:
if False:
    rf_study = optuna.create_study()
    rf_study.optimize(rf_objective, n_trials=50,show_progress_bar=True)
    best_rf_params = rf_study.best_trial.params

    print('Best RF hyper parameters:', best_rf_params)

In [ ]:
rfParams = {'n_estimators': 127, 'max_depth': 11, 'min_samples_split': 242, 'min_samples_leaf': 177, 'max_features': 0.6247048880385929}

In [ ]:
if False:
    rfTuned,preds = learner(train,test,RandomForestClassifier(**rfParams),num_var,'RandomForestTunedAllCols')

In [ ]:
if False:
    subm(np.mean(preds,axis=0)[:,1],'rfTunedAllFeatures')

In [ ]:
if False:
    fi = rf_feat_importance(rfTuned, train[num_var])
    fi[:10]
    plot_fi(fi[:30]);

In [ ]:
if False:
    to_keep = fi[fi.imp>0.01].cols
    to_keep

In [ ]:
if False:
    rfTuned,preds = learner(train,test,RandomForestClassifier(**rfParams),to_keep,'RandomForestTunedBestCols-0.01')

In [ ]:
if False:
    subm(np.mean(preds,axis=0)[:,1],'RandomForestTunedBestCols-0.01')

In [ ]:
if False:
    np.mean(preds,axis=0)[:,1].shape

In [ ]:
if False:
    xgb_model = XGBClassifier()

In [ ]:
if False:
    xgb1,preds = learner(train,test,xgb_model,num_var,'xgbBasicAllCols')

In [ ]:
def xgb_objective(trial):

        # Set the hyperparameters of the XGBoost classifier.
        params = {'objective':'binary:logistic', 
                  'n_estimators': trial.suggest_int('n_estimators', 20, 300, 30),
                  'learning_rate':trial.suggest_float('learning_rate',0.01,0.3),
                  'max_depth': trial.suggest_int('max_depth', 2, 15),
                  'gamma':trial.suggest_float('gamma',0.1,10),
                  'random_state':0,
                  'subsample': trial.suggest_discrete_uniform('subsample', 0.6, 1.0, 0.05),
                 }
        
        xgb_auc_score_avg = 0
        for idx, (train_idx,val_idx) in enumerate(skfold.split(X,y)):
            train_X = X.iloc[train_idx]
            val_X = X.iloc[val_idx]
            train_y = y[train_idx]
            val_y = y[val_idx]

            xgb_model = XGBClassifier(**params)

            xgb_model.fit(train_X,train_y)

            xgb_prediction = xgb_model.predict_proba(val_X)[:,1]
            xgb_auc_score = auc_score(val_y, xgb_prediction)
#             print(f'The AUC score evaluated on the validation subset using XGB model for fold {idx}: ', xgb_auc_score)

            xgb_auc_score_avg += xgb_auc_score
            
        xgb_auc_score_avg /=nfolds
        print(f'The averaged AUC score evaluated on the validation subset using XGB model:', xgb_auc_score_avg)
        return -xgb_auc_score_avg

In [ ]:
if False:
    xgb_study = optuna.create_study()
    xgb_study.optimize(xgb_objective, n_trials=100,show_progress_bar=True)
    best_xgb_params = xgb_study.best_trial.params

    print('Best XGB hyper parameters:', best_xgb_params)

In [ ]:
bestXgbParams = {'objective':'binary:logistic', 
              'n_estimators':260,
              'learning_rate':0.07028863832271189,
              'max_depth':3,
              'gamma':2.1409782911097133,
              'subsample': 0.65,
              'random_state':0}
xgbTuned = XGBClassifier(**bestXgbParams)

In [ ]:
if False:
    xgbTuned,preds2 = learner(train,test,xgbTuned,num_var,'xgbTunedAllCols')

In [ ]:
if False:
    subm(np.mean(preds,axis=0)[:,1],'xgbTunedAllFeatures')

In [ ]:
if False:
    lgb1,pred = learner(train,test,LGBMClassifier(),num_var,'lgbBaselineAllCols')

In [ ]:
if False:
    subm(np.mean(pred,axis=0)[:,1],'lgbBaselineAllFeatures')

In [ ]:
def lgb_objective(trial):

        # Set the hyperparameters of the XGBoost classifier.
        params = {
        # "device_type": trial.suggest_categorical("device_type", ['gpu']),
            "n_estimators": trial.suggest_categorical("n_estimators", [10000]),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
            "num_leaves": trial.suggest_int("num_leaves", 10, 300),
            "max_depth": trial.suggest_int("max_depth", 3, 12),
            "verbose":-1,
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 1000),
            "reg_alpha": trial.suggest_float("reg_alpha", 0, 15),
            "reg_lambda": trial.suggest_float("reg_lambda", 0, 15),
            "min_split_gain": trial.suggest_float("min_split_gain", 0, 15),
            "subsample": trial.suggest_float(
                "subsample", 0.2, 0.95, step=0.1
            ),
            "subsample_freq": trial.suggest_categorical("subsample_freq", [1]),
            "colsample_bytree": trial.suggest_float(
                "colsample_bytree", 0.2, 0.95, step=0.1
            ),
        }
            
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1121218)

        cv_scores = np.empty(5)
        
        for idx, (train_idx,val_idx) in enumerate(cv.split(X,y)):
            train_X = X.iloc[train_idx]
            val_X = X.iloc[val_idx]
            train_y = y[train_idx]
            val_y = y[val_idx]

            lgb_model = LGBMClassifier(objective='binary',**params)

            lgb_model.fit(train_X,
                          train_y,
                          eval_set=[(val_X,val_y)],
                          eval_metric='auc',
                          early_stopping_rounds=100,
                          callbacks=[
                              LightGBMPruningCallback(trial,'auc')
                          ],
                         )

            lgb_prediction = lgb_model.predict_proba(val_X)[:,1]
            cv_scores[idx] = auc_score(val_y, lgb_prediction)

        return np.mean(cv_scores)

In [ ]:
if False:
    lgbstudy = optuna.create_study(direction="maximize", study_name="LGBM Classifier")
    lgbstudy.optimize(lgb_objective, n_trials=200,show_progress_bar=True)
    best_lgb_params = lgbstudy.best_trial.params

    print('Best LGB hyper parameters:', best_lgb_params)

In [ ]:
bestLgbParams =  {'n_estimators': 10000, 'learning_rate': 0.2512493107313411, 'num_leaves': 116, 'max_depth': 5, 'min_child_samples': 858, 'reg_alpha': 9.29234518903945, 'reg_lambda': 8.49934143927119, 'min_split_gain': 2.223046279087342, 'subsample': 0.7, 'subsample_freq': 1, 'colsample_bytree': 0.9}
# {'n_estimators': 10000, 'learning_rate': 0.1382080070022938, 'num_leaves': 202, 'max_depth': 12, 'min_child_samples': 168, 'reg_alpha': 13.476220865169111, 'reg_lambda': 8.275556475452264, 'min_split_gain': 0.9460923477745282, 'subsample': 0.7, 'subsample_freq': 1, 'colsample_bytree': 0.9}
lgbTuned = LGBMClassifier(**bestLgbParams)

In [ ]:
lgbTuned,predLgb = learner(train,test,lgbTuned,num_var,'lgbTuned3AllCols')

In [ ]:
if False:
    subm(np.mean(predLgb,axis=0)[:,1],'lgbTuneAllFeatures2')

In [ ]:
fi = rf_feat_importance(lgbTuned, train[num_var])
fi[:10]
plot_fi(fi[:30]);

In [ ]:
fi.head(1).cols.to_list()

In [ ]:
best_score = 0
best_feature_num = 0
for i in range(1,fi.shape[0]):
    lgbTuned,predLgb,score = learner(train,test,lgbTuned,fi.head(i).cols.to_list(),f'lgbTuned{i}')
    if score > best_score:
        best_score = score
        best_feature_num = i

In [ ]:
lgbTunedF,predLgb,score = learner(train,test,lgbTuned,fi.head(13).cols.to_list(),f'lgbTuned13F')

In [ ]:
subm(np.mean(predLgb,axis=0)[:,1],'lgbTuned13F')